<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

En este notebook se realiza una exploración exhaustiva de los hiperparámetros del meta-modelo secuencial aplicado al conjunto de datos sobre la enfermedad de Parkinson.

Mediante validación cruzada con 5 particiones, se evalúan diferentes combinaciones de parámetros clave como el número de estimadores, la tasa de aprendizaje y la profundidad máxima de los árboles base.

Además del coeficiente de determinación R², se incluyen métricas complementarias como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE) para proporcionar una evaluación más completa del rendimiento.

El objetivo es identificar las configuraciones que maximicen la capacidad predictiva y la robustez del modelo, fundamentando así las decisiones de diseño y optimización.

## 1. Importación de librerías y configuración del entorno

- Se añade la ruta `../../src` para importar el módulo personalizado `sequential_ensemble.py` que contiene la implementación del meta-modelo secuencial.
- Se importan las librerías necesarias para el procesamiento de datos (`numpy`, `pandas`), modelado (`DecisionTreeRegressor`), preprocesamiento (`StandardScaler`), validación cruzada (`cross_validate`, `KFold`), y métricas de evaluación (`mean_absolute_error`, `mean_squared_error`).
- Finalmente, se importa la clase `SequentialEnsembleRegressor` para utilizar el meta-modelo en los experimentos.

In [1]:
# Importación de librerías y configuración del entorno
import sys
sys.path.append('../../src')

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento de datos

- Se carga el conjunto de datos `parkinsons.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) de la variable objetivo (`y`), que en este caso es la columna `total_UPDRS`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler`, para normalizar su escala y facilitar el entrenamiento del modelo.


In [2]:
# Carga y preprocesamiento de datos
df = pd.read_csv("../../data/parkinsons.csv")
X = df.drop(columns=["total_UPDRS"])
y = df["total_UPDRS"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Definición del espacio de búsqueda de hiperparámetros

- Se establecen los rangos de valores para los hiperparámetros clave a explorar:
  - `n_estimators`: número de modelos base en el ensamblador, con valores 10, 30 y 50.
  - `learning_rate`: tasa de aprendizaje que modula la contribución de cada modelo, con valores 0.05, 0.1 y 0.2.
  - `max_depth`: profundidad máxima permitida para los árboles de decisión, con valores 3, 4 y 5.
- Se inicializa una lista vacía `results` para almacenar los resultados de cada combinación probada durante la experimentación.

In [3]:
# Definimos rangos para los hiperparámetros más relevantes
n_estimators_options = [10, 30, 50]
learning_rate_options = [0.05, 0.1, 0.2]
max_depth_options = [3, 4, 5]

# Para guardar resultados
results = []

## 4. Ejecución de experimentos con validación cruzada

- Se configura la validación cruzada con 5 particiones (`KFold`), mezclando aleatoriamente los datos para mayor representatividad.
- Se realiza un bucle anidado para probar todas las combinaciones posibles de los hiperparámetros definidos: número de estimadores, tasa de aprendizaje y profundidad máxima.
- Para cada configuración, se crea una instancia del meta-modelo secuencial con esos parámetros.
- Se ejecuta la validación cruzada calculando múltiples métricas:  
  - Coeficiente de determinación R²,  
  - Error Absoluto Medio (MAE),  
  - Raíz del Error Cuadrático Medio (RMSE).
- Los resultados (medias y desviaciones estándar) se almacenan en la lista `results` para su posterior análisis.
- Se imprime un resumen en cada iteración para seguimiento en tiempo real de los experimentos.

In [4]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar como diccionario para evitar NaN
scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}

# Bucle para probar combinaciones con validación cruzada
for n_estimators in n_estimators_options:
    for lr in learning_rate_options:
        for max_depth in max_depth_options:
            modelo = SequentialEnsembleRegressor(
                base_estimator=DecisionTreeRegressor,
                n_estimators=n_estimators,
                lr=lr,
                sample_size=0.75,
                est_params={"max_depth": max_depth},
                random_state=42
            )
            scores = cross_validate(modelo, X_scaled, y, cv=kf, scoring=scoring)
            
            mean_r2 = np.mean(scores['test_r2'])
            mean_mae = -np.mean(scores['test_mae'])
            mean_rmse = np.sqrt(-np.mean(scores['test_mse']))
            std_r2 = np.std(scores['test_r2'])
            
            results.append({
                "n_estimators": n_estimators,
                "learning_rate": lr,
                "max_depth": max_depth,
                "mean_r2": mean_r2,
                "std_r2": std_r2,
                "mean_mae": mean_mae,
                "mean_rmse": mean_rmse
            })
            print(f"Evaluado n_estimators={n_estimators}, lr={lr}, max_depth={max_depth} -> R2 mean: {mean_r2:.4f}")

Evaluado n_estimators=10, lr=0.05, max_depth=3 -> R2 mean: -2.3729
Evaluado n_estimators=10, lr=0.05, max_depth=4 -> R2 mean: -2.3047
Evaluado n_estimators=10, lr=0.05, max_depth=5 -> R2 mean: -2.2454
Evaluado n_estimators=10, lr=0.1, max_depth=3 -> R2 mean: -0.5055
Evaluado n_estimators=10, lr=0.1, max_depth=4 -> R2 mean: -0.3915
Evaluado n_estimators=10, lr=0.1, max_depth=5 -> R2 mean: -0.3092
Evaluado n_estimators=10, lr=0.2, max_depth=3 -> R2 mean: 0.4408
Evaluado n_estimators=10, lr=0.2, max_depth=4 -> R2 mean: 0.5606
Evaluado n_estimators=10, lr=0.2, max_depth=5 -> R2 mean: 0.6590
Evaluado n_estimators=30, lr=0.05, max_depth=3 -> R2 mean: 0.1279
Evaluado n_estimators=30, lr=0.05, max_depth=4 -> R2 mean: 0.2501
Evaluado n_estimators=30, lr=0.05, max_depth=5 -> R2 mean: 0.3433
Evaluado n_estimators=30, lr=0.1, max_depth=3 -> R2 mean: 0.5712
Evaluado n_estimators=30, lr=0.1, max_depth=4 -> R2 mean: 0.7024
Evaluado n_estimators=30, lr=0.1, max_depth=5 -> R2 mean: 0.7876
Evaluado n_es

## 5. Análisis y visualización de resultados

- Se convierte la lista de resultados en un DataFrame de Pandas para facilitar su manipulación.
- Se ordenan las configuraciones probadas según el valor medio del coeficiente de determinación R², de mayor a menor, para identificar rápidamente las mejores combinaciones.
- Se reinicia el índice del DataFrame para mantener la consistencia tras el ordenamiento.
- Finalmente, se muestra la tabla con todas las configuraciones evaluadas y sus respectivos resultados, facilitando el análisis comparativo.

In [5]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="mean_r2", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,n_estimators,learning_rate,max_depth,mean_r2,std_r2,mean_mae,mean_rmse
0,50,0.20,5,0.876460,0.020460,2.725617,3.724442
1,30,0.20,5,0.853124,0.022568,3.037966,4.056581
2,50,0.10,5,0.845298,0.022092,3.164647,4.169317
3,50,0.20,4,0.839580,0.020764,3.206472,4.242410
4,30,0.20,4,0.795282,0.022961,3.710773,4.794681
5,30,0.10,5,0.787620,0.020626,3.808965,4.889040
6,50,0.10,4,0.781284,0.016165,3.907464,4.960381
7,50,0.20,3,0.747726,0.019844,4.194685,5.330268
8,50,0.05,5,0.738859,0.021750,4.235989,5.423877
9,30,0.10,4,0.702449,0.023363,4.605276,5.786783


## 6. Interpretación de resultados y conclusiones

Los experimentos realizados muestran cómo el rendimiento del meta-modelo secuencial varía significativamente según la configuración de los hiperparámetros.

- Cuando se utiliza un número bajo de estimadores (`n_estimators=10`) junto con una tasa de aprendizaje baja (`lr=0.05`), el modelo obtiene valores negativos de R², indicando un desempeño peor que una predicción constante basada en la media. Esto refleja que la capacidad del modelo es insuficiente para capturar patrones relevantes bajo esas condiciones.

- A medida que aumentamos el número de estimadores y la tasa de aprendizaje, el modelo mejora su capacidad predictiva, alcanzando valores positivos y elevados de R². Por ejemplo, con 50 estimadores y una tasa de aprendizaje de 0.2, junto con árboles más profundos (`max_depth=5`), el modelo logra un R² promedio de 0.8764, lo que indica que explica aproximadamente el 88% de la variabilidad en los datos de prueba.

- La profundidad máxima de los árboles también influye positivamente en el rendimiento, permitiendo que cada estimador base capture relaciones más complejas, aunque debe vigilarse para evitar sobreajuste.

- Los resultados evidencian un equilibrio importante entre la tasa de aprendizaje y el número de estimadores: una tasa muy baja puede requerir más estimadores para lograr buen desempeño, mientras que una tasa más alta permite mejoras rápidas, siempre que el modelo no se sobreajuste.

- Las métricas complementarias, como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE), confirman estas conclusiones, mostrando que las configuraciones óptimas producen predicciones con errores absolutos y desviaciones reducidas, lo que indica alta precisión y estabilidad.

En resumen, la experimentación confirma que configurar adecuadamente estos hiperparámetros es clave para maximizar la eficacia del meta-modelo secuencial, y que la combinación óptima en este caso corresponde a un número elevado de estimadores, una tasa de aprendizaje moderada-alta y árboles base suficientemente profundos.
